In [66]:
import pandas as pd
import numpy as np
import networkx as nx
import pickle
from typing import Dict, List, Optional, Set, Tuple
import re

from rule_engine import extract_rules, summarise_rules, KGRuleEngine

In [14]:
def load_graph(filepath):
    with open(filepath,'rb') as f:
        G = pickle.load(f)
    return G

In [36]:
CAUSAL_SIGN = {
    "increases":            +1,
    "directly_increases":   +1,
    "positive_correlation": +1,
    "decreases":            -1,
    "directly_decreases":   -1,
    "negative_correlation": -1,
}
AD_PATHOLOGY_NODES = {
    'path(MESH:"Alzheimer Disease")',
    'path(EFO:"late-onset Alzheimers disease")',
}
# BPs whose names contain these keywords are AD-relevant
AD_BP_KEYWORDS = {
    "amyloid", "tau", "neurofibrill", "alzheimer", "neuroinflam",
    "synap", "neuron death", "apoptosis", "oxidative stress",
    "mitochondri", "lipid peroxid", "memory", "cognitive",
    "microglial", "neurodegenerat", "autophagy", "ubiquitin",
}
NAME_RE = re.compile(r'"([^"]+)"')

def _node_name(node) -> str:
    m = NAME_RE.search(str(node))
    return m.group(1) if m else str(node)


def _edge_sign(rel: str) -> Optional[int]:
    rel = rel.lower()
    if rel.startswith("rev_"):
        rel = rel[4:]
    return CAUSAL_SIGN.get(rel)


def _is_gene_node(node) -> bool:
    s = str(node)
    return s.startswith(("g(", "p(", "r("))


def _is_bp_node(node) -> bool:
    return str(node).startswith("bp(")


def _is_ad_bp(node) -> bool:
    name = _node_name(node).lower()
    return any(kw in name for kw in AD_BP_KEYWORDS)


def _build_directed_graph(G) -> nx.DiGraph:
    DG = nx.DiGraph()
    for src, tgt, rel, _ in G.edges(keys=True, data=True):
        sign = _edge_sign(rel)
        if sign is None:
            continue
        if rel.lower().startswith("rev_"):
            src, tgt = tgt, src
        DG.add_edge(src, tgt, sign=sign, rel=rel)
    return DG

In [63]:
kg_path = "../data/KG/ad_network_with_reverse_edges.pkl"
G_ad = load_graph(kg_path)
DG_ad = _build_directed_graph(G_ad)

exp_path = '../data/ANMerge/ANMerge_exp_cleaned.csv'
exp_df = pd.read_csv(exp_path, index_col=0)
expr_genes = set(exp_df.columns)

rules = extract_rules(kg_path = kg_path, expression_genes = expr_genes)
df_rules = summarise_rules(rules)
df_rules

  Direct gene→AD rules:    178
  Pathway activation rules: 8
  Causal chain rules:      553


,name,type,conclusion,n_genes,genes,bp,confidence,evidence
0,chain_SORL1_APP_CTL,direct_gene_ad,CTL,1,SORL1,,0.80,"g(HGNC:""SORL1"") → APP → Alzheimer Disease (net..."
1,chain_MME_APP_CTL,direct_gene_ad,CTL,1,MME,,0.80,"p(HGNC:""MME"") → APP → Alzheimer Disease (net s..."
2,chain_CTSD_APP_AD,direct_gene_ad,AD,1,CTSD,,0.80,"g(HGNC:""CTSD"",var(""C,224,T"")) → APP → Alzheime..."
3,direct_BIN1_AD,direct_gene_ad,AD,1,BIN1,,1.00,"g(HGNC:""BIN1"") --rev_positive_correlation--> p..."
4,direct_PICALM_AD,direct_gene_ad,AD,1,PICALM,,1.00,"g(HGNC:""PICALM"") --rev_positive_correlation-->..."
...,...,...,...,...,...,...,...,...
734,chain_AKAP13_4hops_AD,causal_chain,AD,1,AKAP13,,0.60,AKAP13 → MAPT → APP → Alzheimer Disease
735,chain_VPS35_5hops_AD,causal_chain,AD,1,VPS35,,0.45,VPS35 → APP → APP → TNF → Alzheimer Disease
736,chain_VPS35_5hops_CTL,causal_chain,CTL,1,VPS35,,0.45,VPS35 → APP → APP → nitric oxide → Alzheimer D...
737,chain_VPS35_4hops_AD,causal_chain,AD,1,VPS35,,0.60,VPS35 → APP → APP → Alzheimer Disease


## Evaluate rules

In [96]:
print("\nComputing eCDF scores...")
ecdf_path = "../EdgeAssignmentTask/neighborhood_selection/results/aggregator2/ANMerge/AD_KG/ecdf_5/sample_scoring_ecdf.csv"
ecdf = pd.read_csv(ecdf_path, index_col=0)
ctl_ids = list(ecdf[ecdf['label']==0].index)

exp = exp_df.T
ecdf_df = compute_ecdf_scores(exp, ctl_ids)
ecdf_df


Computing eCDF scores...


,DCR00001_1,DCR00004_1,DCR00009_1,DCR00013_1,DCR00014_1,DCR00015_1,DCR00018_1,DCR00020_1,DCR00021_1,DCR00025_1,...,TLSMCI602_1,TLSMCI603_1,TLSMCI604_1,TLSMCI605_1,TLSMCI606_1,TLSMCI607_1,TLSMCI608_1,TLSMCI609_1,TLSMCI611_1,TLSMCI612_1
GIMAP4,-0.230924,0.026104,-0.126506,-0.194779,-0.114458,-0.098394,0.014056,-0.234940,-0.371486,-0.443775,...,0.050201,0.050201,0.154618,0.291165,0.146586,0.303213,0.094378,0.014056,-0.222892,0.098394
C1QBP,-0.423695,-0.355422,-0.307229,-0.403614,-0.279116,-0.226908,-0.182731,-0.246988,0.351406,0.471888,...,-0.058233,-0.118474,-0.014056,-0.086345,0.158635,0.255020,-0.138554,-0.287149,-0.467871,-0.034137
OCIAD2,-0.500000,-0.467871,-0.403614,-0.447791,-0.275100,-0.355422,-0.371486,-0.202811,-0.230924,-0.411647,...,0.074297,0.114458,0.086345,0.134538,0.122490,0.455823,0.046185,-0.062249,-0.479920,0.046185
ERICH1,-0.106426,-0.002008,-0.090361,-0.255020,-0.182731,-0.162651,0.014056,-0.094378,-0.255020,-0.150602,...,0.226908,0.046185,0.198795,0.122490,0.367470,0.182731,0.174699,0.006024,-0.090361,0.439759
MRTFA,-0.299197,-0.371486,-0.327309,-0.182731,-0.251004,-0.323293,-0.423695,-0.323293,-0.002008,0.070281,...,0.403614,0.471888,0.419679,0.419679,0.359438,0.142570,0.463855,-0.471888,-0.206827,0.471888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SLC7A7,-0.078313,0.014056,0.022088,-0.082329,-0.259036,-0.126506,-0.062249,-0.042169,-0.435743,-0.455823,...,0.435743,0.295181,0.375502,0.435743,0.467871,0.122490,0.403614,-0.279116,-0.138554,0.303213
NAA10,-0.315261,-0.463855,-0.331325,-0.487952,-0.303213,-0.391566,-0.255020,-0.110442,0.443775,0.295181,...,0.251004,0.130522,0.251004,0.010040,0.066265,0.010040,0.038153,-0.339357,-0.198795,0.038153
ORAI1,-0.238956,-0.319277,-0.339357,-0.178715,-0.214859,-0.399598,-0.467871,-0.399598,0.126506,-0.162651,...,0.483936,0.467871,0.098394,0.487952,0.090361,0.435743,0.134538,-0.403614,-0.307229,0.218876
ZZeqdeed69UrlI7lNY,-0.259036,-0.411647,-0.038153,-0.423695,-0.343373,-0.311245,-0.427711,-0.363454,0.391566,0.283133,...,0.150602,0.246988,0.058233,0.210843,0.062249,0.086345,0.126506,-0.166667,-0.198795,0.226908


In [97]:
print("Fitting rule engine...")
engine = KGRuleEngine(rules, threshold_std=1.0)
engine.fit(ecdf_df, ctl_ids)

print("Evaluating rules on all patients...")
results = engine.predict(ecdf_df, min_fired_rules=1)
results

Fitting rule engine...
Evaluating rules on all patients...


,patient_id,rules_fired,ad_rules,ctl_rules,ad_rate,ctl_rate,conclusion,top_ad_rules,top_ctl_rules
0,DCR00001_1,37,19,18,0.00893,0.01345,CTL,chain_SNCA_MAPT_AD; chain_IL10_IL6_AD; chain_I...,chain_MMP9_APP_CTL; chain_IL10_inflammatory re...
1,DCR00004_1,34,18,16,0.00899,0.01457,CTL,chain_SNCA_MAPT_AD; chain_IL10_IL6_AD; chain_I...,chain_IL10_inflammatory respons_CTL; chain_FOX...
2,DCR00009_1,59,41,18,0.02478,0.01509,AD,chain_HDAC2_memory_AD; direct_PTGS1_AD; chain_...,chain_HDAC2_H3F3A_CTL; chain_IL10_inflammatory...
3,DCR00013_1,26,17,9,0.00679,0.00457,AD,chain_MARCKS_APP_AD; chain_IL23A_inflammatory ...,direct_TNFRSF1B_CTL; chain_MARCKS_5hops_CTL; c...
4,DCR00014_1,17,8,9,0.00510,0.01101,CTL,pw_suppress_mitochondria-nucleus signaling pat...,chain_FOXO4_insulin receptor sig_CTL; direct_T...
...,...,...,...,...,...,...,...,...,...
686,TLSMCI607_1,423,229,194,0.15592,0.17723,CTL,direct_BIN1_AD; chain_CYLD_Wnt signaling pathw...,chain_SORL1_APP_CTL; chain_RTN4_APP_CTL; chain...
687,TLSMCI608_1,531,299,232,0.25206,0.28756,CTL,chain_CTSD_APP_AD; direct_PICALM_AD; direct_CD...,chain_SORL1_APP_CTL; chain_MME_APP_CTL; chain_...
688,TLSMCI609_1,39,22,17,0.00941,0.01349,CTL,chain_SNCA_MAPT_AD; chain_IL23A_inflammatory r...,chain_FOXO4_insulin receptor sig_CTL; direct_T...
689,TLSMCI611_1,36,21,15,0.01188,0.01462,CTL,chain_SNCA_MAPT_AD; chain_IL23A_inflammatory r...,chain_IL10_inflammatory respons_CTL; chain_FOX...


In [98]:
print("\nRule-based prediction summary:")
print(results["conclusion"].value_counts().to_string())

ecdf['label'] = ecdf['label'].map({1:'AD',0:'CTL'})
true_labels = ecdf['label']
results["true"] = true_labels.values

correct = (results["conclusion"] == results["true"]) & (results["conclusion"] != "uncertain")
covered = results["conclusion"] != "uncertain"
print(f"\nCoverage (patients with ≥1 rule fired): {covered.sum()} / {len(results)}")
if covered.sum() > 0:
    acc = correct.sum() / covered.sum()
    print(f"Accuracy on covered patients: {acc:.3f}")


Rule-based prediction summary:
conclusion
CTL    448
AD     243

Coverage (patients with ≥1 rule fired): 691 / 691
Accuracy on covered patients: 0.454


## ecdf score
+ empirical Culmulative Distribution:
+ empirical Culmulative Distribution Function:
+ this implementation:
+ Vinay implementation:

In [91]:
def compute_ecdf_scores(
    expression_df: pd.DataFrame,
    ctrl_columns: List[str],
) -> pd.DataFrame:
    """Per-gene eCDF scores for all patients using the control distribution.

    For each gene g and patient p:
        ecdf_score[g, p] = F_ctrl(x_{g,p}) - 0.5

    where F_ctrl is the empirical CDF fitted on ctrl_columns patients.
    Score is in [-0.5, 0.5]: 0 = at control median, +0.5 = above all controls,
    -0.5 = below all controls.

    Uses searchsorted for O(n_genes * n_ctrl * log(n_ctrl)) complexity.
    """
    expr = expression_df.copy()
    expr = expr[~expr.index.duplicated(keep='first')]

    ctrl_cols = [c for c in ctrl_columns if c in expr.columns]
    ctrl_expr = expr[ctrl_cols].values  # (n_genes, n_ctrl)
    all_expr = expr.values              # (n_genes, n_patients)
    n_genes, n_ctrl = ctrl_expr.shape
    n_patients = all_expr.shape[1]

    ecdf_matrix = np.empty((n_genes, n_patients), dtype=float)
    for g in range(n_genes):
        ctrl_sorted = np.sort(ctrl_expr[g])
        ecdf_matrix[g] = np.searchsorted(ctrl_sorted, all_expr[g], side='right') / n_ctrl

    score_matrix = ecdf_matrix - 0.5
    return pd.DataFrame(score_matrix, index=expr.index, columns=expr.columns)
